# Pastrimi i të Dhënave

Hapat: rreshti i fundit për çdo vend → mean imputation → CFR + Cases_per_100k → ruajtja

## 1. Ngarkimi

In [ ]:
import pandas as pd
import numpy as np

COUNTRIES = [
    'Albania', 'Kosovo', 'North Macedonia', 'Bosnia and Herzegovina',
    'Montenegro', 'Croatia', 'Slovenia', 'Bulgaria', 'Romania',
    'Hungary', 'Austria', 'Italy', 'Greece', 'Germany',
    'France', 'Spain', 'Portugal', 'Poland', 'Czechia', 'Switzerland'
]

COLUMNS = [
    'location', 'date', 'total_cases', 'total_deaths',
    'total_vaccinations_per_hundred', 'gdp_per_capita', 'population'
]

df_raw = pd.read_csv('../data/raw/covid_data.csv', usecols=COLUMNS, parse_dates=['date'])
df = df_raw[df_raw['location'].isin(COUNTRIES)].sort_values(['location', 'date'])
print(f'Dataset i ngarkuar: {df.shape}')

## 2. Rreshti i Fundit për Çdo Vend

In [ ]:
# Merr datën maksimale (të fundit) për çdo vend
df_latest = df.groupby('location', as_index=False).last().reset_index(drop=True)
print(f'Rreshtat pas filtrimit: {len(df_latest)} (1 për çdo vend)')
df_latest[['location', 'date', 'total_cases', 'total_deaths']]

## 3. NaN Para Pastrimit

In [ ]:
print('NaN para pastrimit:')
print(df_latest.isnull().sum())

## 4. Mean Imputation

In [ ]:
NUMERIC_COLS = ['total_cases', 'total_deaths', 'total_vaccinations_per_hundred', 'gdp_per_capita']

df_clean = df_latest.copy()
for col in NUMERIC_COLS:
    nan_count = df_clean[col].isnull().sum()
    if nan_count > 0:
        mean_val = df_clean[col].mean()
        df_clean[col] = df_clean[col].fillna(mean_val)
        print(f'{col}: {nan_count} NaN → zëvendësuar me {mean_val:.2f}')

print(f'\nNaN pas pastrimit: {df_clean.isnull().sum().sum()}')

## 5. Llogaritja e CFR dhe Cases_per_100k

In [ ]:
df_clean['CFR'] = (df_clean['total_deaths'] / df_clean['total_cases'] * 100).round(2)
df_clean['Cases_per_100k'] = (df_clean['total_cases'] / df_clean['population'] * 100_000).round(2)

df_clean[['location', 'total_cases', 'total_deaths', 'CFR', 'Cases_per_100k']]

## 6. Ruajtja e Dataset-it të Pastruar

In [ ]:
output_path = '../data/processed/covid_clean.csv'
df_clean.to_csv(output_path, index=False)
print(f'Dataset i pastruar ruajtur: {output_path}')
print(f'Shape: {df_clean.shape}')